<a href="https://colab.research.google.com/github/brais-brais/CamposDeFutbol/blob/main/guia_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#FUNCIONAMIENTO DE LOS AGENTES

##1. INTRODUCCIÓN

El objetivo de este documento es explicar el funcionamiento de un sistema con agentes. Para ello, se describirán algunos conceptos fundamentales y se mostrará de manera práctica el comportamiento general de estos.

Para alcanzar este objetivo, el primer paso y el más básico de todos es entender qué es un agente. Un agente no es más que la combinación de un cerebro que gestiona todas las operaciones, con unas extremidades que son las que realizan dichas operaciones ordenadas por el cerebro. En esta analogía, el cerebro es el LLM , mientras que las extremidades son códigos, también conocidos como "tools", es decir, herramientas que permiten al modelo desenvolver las acciones que quiere llevar a cabo. Por tanto, el objetivo del agente será actuar de puente entre el cerebro y las extremidades

Ahora que ya sabemos cuáles son los dos pilares fundamentales que conforman un agente, vamos a verlos por separado:
-Un LLM (Large Language Model) no es más que un sistema entrenado inicialmente con unas cantidades enormes de texto o imágenes, y lo que hace es recibir una entrada y devolver una salida. El proceso de entrenamiento previo es muy largo y costoso, ya que se alimenta con cantidades colosales de información filtrada extraída de novelas, repositorios de código (como GitHub) o artículos científicos. Además, hay partes posteriores del entrenamiento más complejas, en las que por ejemplo le propones al agente una situación concreta y debe saber escoger la herramienta correcta. Aunque este trabajo puede tardar semanas o meses en completarse, el resultado final es un modelo con una precisión muy alta.
-Por su parte, las herramientas no son modelos entrenados, sino bloques de código o servicios externos que el agente puede invocar para interactuar con el entorno o realizar cálculos concretos, recibiendo unos parámetros de entrada y devolviendo un resultado. A diferencia del entrenamiento masivo y lento de un LLM, crear y definir una herramienta es rápido y puntual, pero exige especificar con claridad qué hace, qué parámetros espera y qué devuelve, para que el modelo pueda generar la petición de uso adecuada cuando la situación lo requiera. Estas herramientas no forman parte interna del modelo ni del núcleo del agente: son extremidades externas que el código del agente coordina.

De forma similar, el agente también puede controlar mecanismos de memoria externa como RAG (Retrieval-Augmented Generation). En lugar de ejecutar acciones o cálculos, su función es consultar fuentes o bases de conocimiento externas para recuperar información relevante e inyectarla en el prompt, ampliando así la información que se entrega al LLM antes de que genere su respuesta.

##INSTALACIÓN DE DEPENDENCIAS PARA EJECUTAR LOS CÓDIGOS

In [24]:
import importlib.metadata
import subprocess
import sys
import time

MODEL_NAME = "qwen3:8b"
MCP_SPEC = "mcp>=1.27,<2"


def verificar_mcp() -> str:
    """Comprueba que 'mcp' está instalado, es importable y cumple MCP_SPEC."""
    version_str = importlib.metadata.version("mcp")
    partes = [int(p) for p in version_str.split(".")[:2]]
    mayor, menor = partes[0], (partes[1] if len(partes) > 1 else 0)

    if mayor != 1 or menor < 27:
        raise RuntimeError(
            f"Versión de 'mcp' incompatible ({version_str}). Se requiere {MCP_SPEC}."
        )

    import mcp  # Verifica que el paquete se importa sin errores
    return version_str


# Comprobación de si estamos ejecutando en Google Colab
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    print("☁️ Entorno de Google Colab detectado. Ejecutando instalación automática remota...")
    try:
        # 1. Instalar zstd (requerido por el instalador de Ollama)
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)

        # 2. Instalar Ollama en el entorno de Colab
        subprocess.run(
            "curl -fsSL https://ollama.com/install.sh | sh",
            shell=True,
            check=True,
        )

        # 3. Instalar y verificar dependencias de Python con versión acotada de MCP
        print(f"📦 Instalando y verificando {MCP_SPEC} y ollama...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", MCP_SPEC, "ollama"],
            check=True,
        )
        version_mcp = verificar_mcp()
        print(f"✅ Paquete 'mcp' verificado (versión {version_mcp}).")

        # 4. Arrancar Ollama en segundo plano y descargar el modelo configurado
        ollama_process = subprocess.Popen(["ollama", "serve"])
        time.sleep(5)

        subprocess.run(["ollama", "pull", MODEL_NAME], check=True)
        print(f"✅ Ollama arrancado y modelo '{MODEL_NAME}' descargado correctamente.")

    except subprocess.CalledProcessError as e:
        print(f"\n❌ Error durante un comando de instalación (código {e.returncode}): {e}")
        print(f"Si falló pip, prueba:\n  pip install \"{MCP_SPEC}\" ollama")
        sys.exit(1)

    except Exception as e:
        print(f"\n❌ Error al preparar el entorno en Colab: {e}")
        print(f"Para forzar la versión compatible de MCP:\n  pip install --force-reinstall \"{MCP_SPEC}\"")
        sys.exit(1)
else:
    print(f"💻 Entorno local detectado. Verificando dependencias y modelo '{MODEL_NAME}'...")
    entorno_ok = True

    try:
        version_mcp = verificar_mcp()
        import ollama

        response = ollama.list()
        modelos_raw = getattr(response, "models", None) or response.get("models", [])
        nombres_modelos = [
            getattr(m, "model", None) or getattr(m, "name", None) or m.get("model") or m.get("name")
            for m in modelos_raw
        ]

        if MODEL_NAME not in nombres_modelos:
            entorno_ok = False
            print(f"\n⚠️ Falta descargar el modelo '{MODEL_NAME}' en Ollama.")
            print(f"Modelos detectados actualmente: {nombres_modelos or 'ninguno'}.\n")
            print("👉 Qué tienes que hacer:")
            print(f"   1. Ejecuta en tu terminal:  ollama pull {MODEL_NAME}")
            print("   2. Vuelve a ejecutar esta celda.")

    except importlib.metadata.PackageNotFoundError:
        entorno_ok = False
        print("\n⚠️ No se ha encontrado el paquete 'mcp' en este entorno de Python.\n")
        print("👉 Qué tienes que hacer:")
        print(f"   1. Si usas un entorno virtual (venv), actívalo y ejecuta:")
        print(f"      pip install \"{MCP_SPEC}\" ollama")
        print(f"   2. Si usas el Python del sistema (Debian/PEP 668), ejecuta:")
        print(f"      {sys.executable} -m pip install --break-system-packages \"{MCP_SPEC}\" ollama")
        print("   3. Vuelve a ejecutar esta celda.")

    except Exception as e:
        entorno_ok = False
        print(f"\n⚠️ No se pudo completar la verificación local: {e}\n")
        print("👉 Qué tienes que hacer:")
        print(f"   1. Instala o ajusta las dependencias:")
        print(f"      {sys.executable} -m pip install --break-system-packages \"{MCP_SPEC}\" ollama")
        print("   2. Asegúrate de que el servicio de Ollama está arrancado:")
        print("      ollama serve")
        print(f"   3. Descarga el modelo si no lo tienes:")
        print(f"      ollama pull {MODEL_NAME}")
        print("   4. Vuelve a ejecutar esta celda.")

    if entorno_ok:
        print(
            f"✅ Entorno local verificado: 'mcp' ({version_mcp}), 'ollama' y modelo "
            f"'{MODEL_NAME}' disponibles."
        )

☁️ Entorno de Google Colab detectado. Ejecutando instalación automática remota...
📦 Instalando y verificando mcp>=1.27,<2 y ollama...
✅ Paquete 'mcp' verificado (versión 1.30.0).
✅ Ollama arrancado y modelo 'qwen3:8b' descargado correctamente.


#EJEMPLO DEL PUNTO DÉBIL DE LOS LLM
Una vez entendido el funcionamiento de los LLMs, salta a la vista que si realizamos una consulta preguntando por información actual no vamos a obtener una respuesta útil, ya que estos solo contienen información de antes de que acabase su entrenamiento. Veámoslo a continuación:

In [ ]:
import ollama

# Realizamos una consulta al LLM sobre un evento o dato muy reciente
# que esté fuera de su base de conocimientos de entrenamiento.
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {
            'role': 'user',
            'content': '¿Cuál fue el resultado exacto del último partido oficial del Celta de Vigo jugado esta semana y quién marcó los goles?'
        }
    ]
)

print("Respuesta del LLM sin herramientas:")
print(response['message']['content'])

Como hemos podido observar, el LLM no tiene ni la menor idea de cuál es la respuesta a nuestra pregunta, o bien se la inventa o bien responde que no la sabe. Sin embargo, si le preguntamos a chatgpt o gemini contra quién jugará el celta su próximo partido veremos que responder sin problema. Esto se debe a la segunda parte que conforma a los agentes que mencionamos antes, las herramientas. Puede que la IA no tenga la información necesaria para responder, y es precisamente por eso por lo que queremos que tenga la capacidad de consultar bases de datos, acceder a internet, mandar mensajes y muchas más cosas.

Antes era necesario un código específico para conectar cada modelo de lenguaje con una base de datos o una API, lo que hacía muy difícil e ineficiente conseguir una IA con acceso a múltiples herramientas externas. Para solucionar este problema, se creó MCP (Model Context Protocol), que sirve para unificar la manera en la que se conectan todos los LLMs con distintos servidores de herramientas, facilitando así la implementación de agentes.

En este contexto, un MCP server es un servidor que tiene un conjunto de herramientas a su cargo, que sabe ejecutarlas y que habla el protocolo MCP para informar de cuáles son las disponibles, recibir peticiones de ejecución y devolver sus respuestas. Más adelante veremos un ejemplo de uno.

##FLUJO DE INTERACCIONES EN UN ENTORNO AGÉNTICO

En este documento vamos a centrarnos en observar el funcionamiento de MCP, acompañando las explicaciones con código para entender todo de una manera más visual. La idea general es que tenemos un agente, que es el que se encarga de conectar las distintas partes del sistema:
- El usuario (nosotros)
- El LLM (un modelo de ollama en nuestro caso)
- Y el MCP server (haremos uno propio)
La siguiente imagen muestra el flujo de funcionamiento de una interacción normal, pero hay que destacar que este flujo es una vez ya se han comunicado distintas partes previamente, lo que implica que el MCP server ya informó al agente previamente de que herramientas tiene disponibles.

<div align="center">
  <img src="https://i.postimg.cc/Y0QKGzQX/MCP.jpg" width="600">
</div>

1 El usuario escribe la petición "realiza la operacion inventada entre 4 y 3" (llamada carton en la figura)

2 El agente recibe la petición y se la reenvía al LLM a través de una API para  que decida que hacer, junto con el historial de conversación y las herramientas disponibles por MCP.

3 Al no disponer de esa información en sus pesos y recibir en el contexto la descripción de las herramientas MCP, la salida más probable que genera el LLM es solicitar al agente la invocación de la operación inventada.

4 El agente obedece y se comunica a través de un canal std con el servidor MCP, usando el protocolo JSON-RPC, indicando que operación quiere realizar y sus parámetros

5 El servidor ejecuta la operación y le devuelve al agente el resultado.

6 El agente repite la consulta anterior, junto con el resultado obtenido de la operación, para que desarrolle una respuesta a partir de él.

7 El LLM le da esa respuesta de vuelta al agente.

8 El agente le muestra finalmente esa respuesta al usuario.

##SERVIDORES MCP

Como vimos previamente, un servidor MCP se trata de un programa que tiene definidas diferentes operaciones que es capaz de realizar, junto con una definición de qué hace cada una, que será la que le llegue al LLM para saber si alguna herramienta del servidor le es útil o no a la hora de atender una solicitud. En lugar de utilizar sockets de red tradicionales o abrir puertos TCP (la alternativa habitual en servidores web), la comunicación por tuberías (stdio) aprovecha la entrada y salida estándar del sistema operativo para que el cliente y el servidor hablen directamente mediante un flujo de procesos hijo, evitando configuraciones de red complejas.

Empezaremos por definir las operaciones que vamos a poder realizar en nuestro servidor. Para hacerlo sencillo, solo habrá una operación posible, llamada op_inv (operación inventada). Esta operación simplemente consiste en realizar unos cálculos entre dos parámetros. Con esto buscamos pedirle al modelo algo que no sabe realizar, puesto que nos lo acabamos de inventar, y por ende tendrá que acudir al servidor MCP para resolver la petición. Es importante aportar una buena definición de para qué sirve nuestra función, ya que esto será lo que el LLM use como input para calcular si tiene que usar esa herramienta para la petición recibida o no.

In [25]:
%%writefile mcp_server.py
#Usamos writefile para crear un fichero .py en el entorno en el que nos encontramos,
#con el siguiente contenido, lo que nos permite invocarlo cuando lo consideremos oportuno.

# Esta librería nos ayuda a controlar el ciclo de vida de nuestro servidor, así
# como a serializar el protocolo, es decir, pasar los mensajes de texto plano
# al formato JSON-RPC, para que el LLM entienda la respuesta. Aunque  podríamos
# programar la gestión de estos mensajes de protocolo desde cero,utilizamos la
# librería oficial de MCP (el SDK) para ahorrarnos todo el código repetitivo de bajo nivel
from mcp.server.fastmcp import FastMCP

# Creamos la instancia del servidor MCP con un nombre identificativo
mcp = FastMCP("NuestroServidor")

# Definimos la función y le añadimos el decorador @mcp.tool()
# para registrarla automáticamente en el servidor MCP como una herramienta
# disponible que el agente puede invocar cuando lo necesite.
@mcp.tool()
def op_inv(a: float, b: float) -> float:
    """Aplica la operación inventada entre dos números a y b.

    Args:
        a: Primer número de la operación.
        b: Segundo número de la operación.
    """
    resultado = (a * b) / 2 + 5
    return resultado

if __name__ == "__main__":
    print("Lanzando el servidor MCP...")
    # Arrancamos el servidor usando comunicación estándar por tuberías (stdio)
    mcp.run(transport="stdio")

Overwriting mcp_server.py


##CÓDIGO AUXILIAR

El siguiente paso será añadir el módulo que implementa una clase auxiliar del agente para comunicarse con el servidor MCP, encapsulando las funciones necesarias para consultar las herramientas disponibles e invocarlas. Desde aquí también se lanzará el servidor MCP como un proceso hijo, mientras que la interacción con Ollama se realizará más adelante en el flujo principal del agente, cuando se utilicen estas funciones tras recibir la respuesta del modelo.

In [26]:
%%writefile mcp_manager.py
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

class MCPManager:
    """
    Clase gestora para administrar la conexión de cliente con un servidor MCP.
    Se encarga de arrancar el proceso del servidor, listar las herramientas
    disponibles y ejecutar las llamadas solicitadas por el LLM.
    """
    def __init__(self, command: str, args: list[str]):
        """
        Inicializa los parámetros necesarios para lanzar y conectar con el servidor MCP.

        Args:
            command: El comando ejecutable (ej. "python3" o ruta a un script).
            args: Los argumentos que se le pasan a ese comando (ej. ["mcp_server.py"]).
        """
        # Configuramos los parámetros de la tubería estándar (stdio) especificando
        # qué comando de sistema debe lanzar y qué script o programa debe ejecutar.
        self.server_params = StdioServerParameters(command=command, args=args)

        # Atributos para almacenar la sesión activa y el contexto de la tubería.
        self.session: ClientSession | None = None
        self._client_context = None

    async def connect(self):
        """
        Inicia el subproceso del servidor en segundo plano y realiza
        el intercambio de protocolos inicial (handshake).
        """
        # 1. Creamos el contexto de comunicación por tuberías (stdio).
        self._client_context = stdio_client(self.server_params)

        # 2. Abrimos el contexto de forma asíncrona para obtener los canales de lectura y escritura.
        read, write = await self._client_context.__aenter__()

        # 3. Inicializamos la sesión del cliente MCP usando esos canales de E/S.
        self.session = ClientSession(read, write)
        await self.session.__aenter__()

        # 4. Ejecutamos la fase de inicialización (negociación de capacidades del protocolo).
        await self.session.initialize()

    async def get_ollama_tools(self) -> list[dict]:
        """
        Consulta al servidor MCP las herramientas que tiene registradas
        y las traduce al formato JSON estructurado que exige Ollama.
        """
        if not self.session:
            raise RuntimeError("El gestor MCP no está conectado.")

        # Pedimos al servidor la lista completa de herramientas disponibles (catálogo).
        mcp_tools = await self.session.list_tools()
        ollama_tools = []

        # Recorremos cada herramienta del servidor para adaptarla al estándar de Ollama.
        for tool in mcp_tools.tools:
            ollama_tools.append({
                "type": "function",
                "function": {
                    "name": tool.name,             # Nombre de la función (ej. "op_inv")
                    "description": tool.description, # Docstring que lee el LLM para saber qué hace
                    "parameters": tool.inputSchema, # Esquema JSON de los argumentos que acepta
                },
            })
        return ollama_tools

    async def call_tool(self, name: str, arguments: dict) -> str:
        """
        Envía la orden de ejecución de una herramienta al servidor
        y devuelve el resultado obtenido.
        """
        if not self.session:
            raise RuntimeError("El gestor MCP no está conectado.")

        # Invocamos la herramienta de forma remota a través de la sesión,
        # enviándole el nombre y los argumentos en formato diccionario.
        result = await self.session.call_tool(name, arguments)

        # Extraemos el texto plano de la respuesta devuelta por el servidor.
        return result.content[0].text

    async def disconnect(self):
        """
        Cierra de forma limpia y ordenada la sesión del cliente
        y detiene el subproceso del servidor.
        """
        if self.session:
            # Cerramos la sesión MCP asíncronamente
            await self.session.__aexit__(None, None, None)
        if self._client_context:
            # Cerramos la tubería stdio y matamos el subproceso del servidor
            await self._client_context.__aexit__(None, None, None)

Overwriting mcp_manager.py


##CÓDIGO DEL AGENTE

En este próximo código será donde realicemos una petición al LLM, e iremos imprimiendo mensajes progresivamente para entender el flujo de interacciones con el servidor MCP.
Podemos observar como en primera instancia nos conectamos al servidor, y gracias a la "traducción" del MCP manager que ejecutamos anteriormente ollama recibe correctamente la herramienta que diseñamos al principio (op_inv). A continuación realizamos la petición, que hace que el LLM le pida al servidor MCP que realice la operación, especificándole los parámetros. Este ejecuta y devuelve el resultado, que ollama recibe y acaba mostrándonos por pantalla.

In [27]:
%%writefile client_runner.py
import asyncio
import sys
import ollama
from mcp_manager import MCPManager

# Definimos el modelo de Ollama que usaremos como cerebro,
# la ruta dinámica del intérprete de Python actual y el script del servidor.
MODEL_NAME = "qwen3:8b"
PYTHON_PATH = sys.executable
SERVER_PATH = "mcp_server.py"

async def main():
    print("**Lanzando y conectando con el servidor MCP...")

    # 1. Instanciamos el gestor pasándole el comando de Python y el script del servidor.
    mcp = MCPManager(command=PYTHON_PATH, args=[SERVER_PATH])

    # 2. Arrancamos el servidor de forma asíncrona mediante la tubería stdio.
    await mcp.connect()

    # 3. Consultamos al servidor las herramientas disponibles y las adaptamos a Ollama.
    tools = await mcp.get_ollama_tools()
    print(f"Herramientas detectadas por el agente: {[t['function']['name'] for t in tools]}")

    # Definimos la petición de ejemplo que le haremos al modelo.
    prompt_usuario = "Por favor, calcula la op_inv usando a=10 y b=4."
    print(f"\nUsuario > {prompt_usuario}")

    # Inicializamos el historial de mensajes de la conversación con el rol del sistema y del usuario.
    messages = [
        {'role': 'system', 'content': 'Eres un asistente útil que usa herramientas cuando las necesita.'},
        {'role': 'user', 'content': prompt_usuario}
    ]

    # Bucle principal de interacción con el LLM (permite manejar varias rondas de llamadas si fuera necesario).
    while True:
        # Enviamos el historial de chat y el catálogo de herramientas (tools) al modelo de Ollama.
        response = ollama.chat(model=MODEL_NAME, messages=messages, tools=tools)

        # Si el modelo responde con texto normal (sin solicitar ninguna herramienta), terminamos el bucle.
        if not response['message'].get('tool_calls'):
            print(f"\nAgente (Respuesta Final) > {response['message']['content']}")
            break

        # Si el modelo decide que necesita usar una herramienta, añadimos su respuesta al historial.
        messages.append(response['message'])

        # Iteramos sobre cada herramienta que el LLM haya decidido invocar.
        for tool_call in response['message']['tool_calls']:
            fn_name = tool_call['function']['name']
            fn_args = tool_call['function']['arguments']

            print(f"\n[MCP] El LLM ha decidido invocar: '{fn_name}' con los argumentos: {fn_args}")

            # 4. Ejecutamos la herramienta de manera remota a través del gestor MCP y recogemos el resultado.
            result_text = await mcp.call_tool(fn_name, fn_args)
            print(f"-> Resultado devuelto por la herramienta de MCP: {result_text}")

            # Añadimos el resultado obtenido al historial de la conversación con el rol 'tool',
            # para que el LLM pueda leerlo en la siguiente iteración del bucle.
            messages.append({
                'role': 'tool',
                'content': f"RESULTADO OFICIAL: {result_text}",
                'name': fn_name
            })

    # 5. Cerramos la sesión y desconectamos el servidor de forma limpia al terminar.
    await mcp.disconnect()

# Punto de entrada estándar para ejecutar funciones asíncronas en Python.
if __name__ == "__main__":
    asyncio.run(main())

Overwriting client_runner.py


##COMPROBACIÓN

Con esta celda comprobaremos que el código de arriba efectivamente muestra el flujo adecuado.

In [29]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

#Definimos una función para asegurar que el modelo de ollama está escuchando
def ollama_activo() -> bool:
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
            return resp.status == 200
    except (urllib.error.URLError, TimeoutError, OSError):
        return False


if not ollama_activo():
    print("⚙️ Ollama no está activo; levantando 'ollama serve'...", flush=True)
    log_f = open("/tmp/ollama.log", "ab")
    # start_new_session=True evita que se cierre al terminar una celda o subproceso
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_f,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(25):
        if ollama_activo():
            print("✅ Ollama listo en http://127.0.0.1:11434", flush=True)
            break
        time.sleep(1)
    else:
        print("❌ Ollama no llegó a responder. Últimas líneas de /tmp/ollama.log:", flush=True)
        with open("/tmp/ollama.log", "r", encoding="utf-8", errors="replace") as f:
            print("".join(f.readlines()[-25:]), flush=True)
        raise RuntimeError("No se pudo iniciar Ollama")

with subprocess.Popen(
    [sys.executable, "-u", "client_runner.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()

**Lanzando y conectando con el servidor MCP...
[11/26/26 16:12:20] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
Herramientas detectadas por el agente: ['op_inv']

Usuario > Por favor, calcula la op_inv usando a=10 y b=4.

[MCP] El LLM ha decidido invocar: 'op_inv' con los argumentos: {'b': 4, 'a': 10}
[11/26/26 16:17:19] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
-> Resultado devuelto por la herramienta de MCP: 25.0

Agente (Respuesta Final) > La operación inventada (op_inv) con los valores a = 10 y b = 4 da como resultado:

**Resultado:** 25.0

¿Necesitas ayuda con algo más? 😊

Proceso terminado con código: 0


##CONCLUSIÓN

Como podemos ver, todo sigue un flujo ordenado de mensajes entre las distintas partes. Pese a que esto solo es una muestra simplificada, a la hora de acceder bases de datos, hacer consultas en la red o ejecutar funciones todo sigue este mismo orden, por lo que ya tenemos una idea general de cómo funciona un agente a la hora de acudir a herramientas.


In [30]:
import importlib.util
import subprocess
import sys

PAQUETES_FRAMEWORK = [
    ("langchain_mcp_adapters", "langchain-mcp-adapters"),
    ("langchain_ollama", "langchain-ollama"),
    ("langgraph", "langgraph"),
]

en_colab = "google.colab" in sys.modules or importlib.util.find_spec("google.colab") is not None

faltantes = [
    pip_name
    for mod_name, pip_name in PAQUETES_FRAMEWORK
    if importlib.util.find_spec(mod_name) is None
]

print(f"🔎 Entorno detectado: {'Google Colab (nube)' if en_colab else 'Local'}")

if not faltantes:
    print("✅ Paquetes opcionales de framework listos: langchain-mcp-adapters, langchain-ollama y langgraph.")
else:
    paquetes_str = " ".join(faltantes)
    print(f"⚠️ No se han encontrado los siguientes paquetes opcionales: {', '.join(faltantes)}")

    if en_colab:
        print(f"⏳ Instalando automáticamente en Google Colab: {paquetes_str}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
        print("✅ Instalación completada en Google Colab.")
    else:
        print("\n👉 Cómo instalarlos en entorno local (Linux/Python):")
        print(f"   {sys.executable} -m pip install --break-system-packages {paquetes_str}")

🔎 Entorno detectado: Google Colab (nube)
⚠️ No se han encontrado los siguientes paquetes opcionales: langchain-mcp-adapters, langchain-ollama
⏳ Instalando automáticamente en Google Colab: langchain-mcp-adapters langchain-ollama...
✅ Instalación completada en Google Colab.


In [31]:
import traceback


def mostrar_subexcepciones(exc: BaseException, nivel: int = 0) -> None:
    prefijo = "  " * nivel
    if isinstance(exc, BaseExceptionGroup):
        print(f"{prefijo}ExceptionGroup: {exc.message} ({len(exc.exceptions)} sub-excepción/es)")
        for idx, sub in enumerate(exc.exceptions, start=1):
            print(f"{prefijo}--- Sub-excepción #{idx} ---")
            mostrar_subexcepciones(sub, nivel + 1)
    else:
        print(f"{prefijo}{type(exc).__name__}: {exc}")
        traceback.print_exception(type(exc), exc, exc.__traceback__)


pregunta_prueba = "Consulta el estado usando la herramienta disponible"

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    loop = None

try:
    if loop and loop.is_running():
        resultado = await ejecutar_con_framework(pregunta_prueba)
    else:
        resultado = asyncio.run(ejecutar_con_framework(pregunta_prueba))

    print("Resultado repr:", repr(resultado))
    print("Resultado:\n", resultado)
except BaseExceptionGroup as eg:
    mostrar_subexcepciones(eg)

NameError: name 'asyncio' is not defined